<a href="https://colab.research.google.com/github/WilsonForeroB/ColabIA/blob/main/CuantizacionVLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Actividad: Cuantiza un LLM y sírvelo con vLLM**

**Duración:** 2 horas · **Nivel:** desarrollador/a con Python básico

En la sesión vimos que **QLoRA = LoRA + comprimir W a 4 bits** (un modelo de 7B pasa de ~14 GB a ~4 GB).
Hoy vamos a hacer esa compresión **nosotros mismos**, sin entrenar nada: vamos a **cuantizar** un modelo
y a servirlo con **vLLM**, el motor de inferencia de alto rendimiento más usado en producción.

**Al terminar sabrás:**
1. Montar un entorno de vLLM en Colab y servir un modelo (Python y API compatible con OpenAI).
2. Explicar qué es FP16, FP8, INT8 e INT4, y la diferencia entre *weight-only* (W4A16) y *weights+activations* (W8A8).
3. Cuantizar un modelo con **4 alternativas**, de la más rápida a la más precisa.
4. Medir el impacto: tamaño en disco, VRAM, velocidad y calidad.

### Agenda (120 min)
| Min | Bloque | Qué haces |
|---|---|---|
| 0–15 | **Paso 0–2** · Entorno | GPU, instalar vLLM, primer modelo (línea base FP16) |
| 15–25 | **Paso 3** · Conceptos | Tabla de formatos y decisión |
| 25–40 | **Alternativa A** · Cuantización al vuelo | `quantization="fp8"` / `bitsandbytes` y modelos ya cuantizados del Hub |
| 40–55 | **Alternativa B** · FP8 dinámico con LLM Compressor | Sin datos de calibración, 2 min |
| 55–85 | **Alternativa C** · INT4 GPTQ / AWQ (W4A16) | Con datos de calibración |
| 85–95 | **Alternativa D** · INT8 W8A8 (SmoothQuant + GPTQ) | La opción para GPUs T4 |
| 95–110 | **Paso 8–9** · Comparar y servir | Tabla de resultados + `vllm serve` |
| 110–120 | **Entrega** | Conclusiones |

> ⏱️ **Si vas justo de tiempo:** haz Paso 0–2, Alternativa A, Alternativa B **o** C, y el Paso 8. El resto es ampliación.

## 🧭 Antes de empezar: elige bien la GPU

`Entorno de ejecución → Cambiar tipo de entorno de ejecución`

| GPU en Colab | Arquitectura | FP8 | INT4 (W4A16) | INT8 (W8A8) | bitsandbytes | Recomendación |
|---|---|---|---|---|---|---|
| **T4** (gratis) | Turing (sm75) | ❌ | ⚠️ kernels de respaldo, lento | ✅ | ✅ | Usa Alternativas A (bnb) y D |
| **L4** (Colab Pro) | Ada (sm89) | ✅ nativo | ✅ Marlin | ✅ | ✅ | **⭐ Ideal para la actividad** |
| **A100** (Colab Pro) | Ampere (sm80) | ⚠️ solo pesos (Marlin) | ✅ Marlin | ✅ | ✅ | Muy bien |
| **H100** | Hopper (sm90) | ✅ nativo | ✅ Machete | ✅ | ✅ | Lo mejor |

> La **cuantización** (crear el modelo comprimido con LLM Compressor) funciona en cualquier GPU.
> Lo que depende de la GPU es si **vLLM** tiene kernels rápidos para **ejecutarlo**.

---
## Paso 0 · Verificar GPU

In [ ]:
# ===========================================================================
# PASO 0: Verificar GPU
# ===========================================================================
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Paso 1 · Instalar vLLM + LLM Compressor

- **vLLM**: motor de inferencia (PagedAttention, batching continuo, kernels cuantizados).
- **LLM Compressor** (`llmcompressor`): librería del proyecto vLLM para **crear** modelos cuantizados (FP8, GPTQ, AWQ, SmoothQuant...). Guarda en formato `compressed-tensors`, que vLLM carga directamente.

Usamos `uv` porque instala en ~1–2 min en lugar de 5–8 con `pip`.

> ⚠️ **Después de instalar, reinicia la sesión** (`Entorno de ejecución → Reiniciar sesión`): vLLM trae su propia versión de PyTorch.

In [ ]:
# ===========================================================================
# PASO 1: Instalación (~2-4 min)
# ===========================================================================
!pip install -q uv
!uv pip install --system -q vllm llmcompressor bitsandbytes datasets openai

import vllm, llmcompressor, torch
print("vLLM:", vllm.__version__, "| llmcompressor:", llmcompressor.__version__, "| torch:", torch.__version__)
print("\n👉 Ahora: Entorno de ejecución → Reiniciar sesión, y continúa en la siguiente celda.")

## Paso 2 · Configuración y script de benchmark

Para no pelearnos con la memoria de la GPU, **cada prueba de vLLM se ejecuta en un proceso aparte**
(`bench_vllm.py`). Al terminar el proceso, la VRAM se libera por completo y podemos cuantizar o probar otro modelo.

**Modelo de la actividad:** `Qwen/Qwen2.5-1.5B-Instruct` (≈3 GB en FP16, cuantiza en pocos minutos).
Si vas muy justo de tiempo usa `Qwen/Qwen2.5-0.5B-Instruct`.

In [ ]:
# ===========================================================================
# PASO 2a: Variables globales (ejecutar tras reiniciar)
# ===========================================================================
import os, torch
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # alternativa rápida: "Qwen/Qwen2.5-0.5B-Instruct"
WORK = "/content/modelos"
os.makedirs(WORK, exist_ok=True)

cc = torch.cuda.get_device_capability()
GPU = torch.cuda.get_device_name()
print(f"GPU: {GPU} | compute capability: sm{cc[0]}{cc[1]}")
print("FP8 nativo:", "✅" if cc >= (8, 9) else "❌ (usa INT8/INT4/bnb)")

# Filtro para ver solo lo importante del log de vLLM (quítalo si algo falla y quieres el log completo)
FILTER = "Model loading took|❓|💬|tag.: |tok_s|load_s|Traceback|Error"

In [ ]:
%%writefile /content/bench_vllm.py
# ===========================================================================
# PASO 2b: Script de benchmark. Uso:
#   python bench_vllm.py --model <ruta_o_id> [--quantization fp8|bitsandbytes] [--tag nombre]
# Imprime: respuestas, tokens/s y VRAM de pesos, y guarda una fila en /content/resultados.jsonl
# ===========================================================================
import argparse, json, time, os, torch
from vllm import LLM, SamplingParams

p = argparse.ArgumentParser()
p.add_argument("--model", required=True)
p.add_argument("--quantization", default=None)
p.add_argument("--tag", default=None)
p.add_argument("--max_model_len", type=int, default=2048)
a = p.parse_args()

PROMPTS = [
    "Explica qué es la cuantización de un modelo de lenguaje en 3 frases.",
    "Escribe una función en Python que devuelva los números primos menores que n.",
    "Si tengo 3 manzanas y compro el doble de las que tengo, y luego me como 2, ¿cuántas tengo? Razona paso a paso.",
    "Traduce al inglés: 'La seguridad de los LLM empieza por validar sus entradas.'",
]

kwargs = dict(model=a.model, max_model_len=a.max_model_len, gpu_memory_utilization=0.85, enforce_eager=True)
if a.quantization:
    kwargs["quantization"] = a.quantization
if torch.cuda.get_device_capability() < (8, 0):
    kwargs["dtype"] = "half"          # T4 no soporta bfloat16

t0 = time.time()
llm = LLM(**kwargs)
load_s = time.time() - t0

sp = SamplingParams(temperature=0.0, max_tokens=256)
msgs = [[{"role": "user", "content": q}] for q in PROMPTS]
llm.chat(msgs[:1], sp)                                # calentamiento

# Throughput: mismas preguntas repetidas en lote (simula varios usuarios)
batch = msgs * 8
t0 = time.time()
outs = llm.chat(batch, sp)
dt = time.time() - t0
ntok = sum(len(o.outputs[0].token_ids) for o in outs)

print("\n" + "=" * 80)
for q, o in zip(PROMPTS, outs[:len(PROMPTS)]):
    print(f"\n❓ {q}\n💬 {o.outputs[0].text.strip()[:600]}")
print("\n" + "=" * 80)
row = dict(tag=a.tag or a.model, model=a.model, quantization=a.quantization,
           load_s=round(load_s, 1), tok_s=round(ntok / dt, 1), gen_tokens=ntok,
           respuesta_mates=outs[2].outputs[0].text.strip()[-200:])
print(json.dumps(row, ensure_ascii=False, indent=2))
print("👀 Busca en el log de arriba la línea 'Model loading took X GiB' → VRAM real de los pesos")
with open("/content/resultados.jsonl", "a") as f:
    f.write(json.dumps(row, ensure_ascii=False) + "\n")

### Paso 2c · Línea base: el modelo original en FP16/BF16

Este es nuestro punto de referencia. **Apunta** en tu tabla:
- `Model loading took X GiB` (aparece en el log)
- `tok_s` (tokens por segundo)
- Si la respuesta de matemáticas es correcta (3 + 6 − 2 = **7**)

In [ ]:
!python /content/bench_vllm.py --model $MODEL_ID --tag "0_base_fp16" 2>&1 | grep -E "$FILTER"

In [ ]:
# Tamaño en disco del modelo original (caché de Hugging Face)
!du -sh ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct* 2>/dev/null

---
## Paso 3 · Conceptos en 10 minutos

### ¿Qué es cuantizar?
Guardar cada peso con **menos bits**. Un peso en FP16 ocupa 16 bits; en INT4, 4 bits → **4× menos memoria**.

```
FP16:  0.0731  -0.2104   0.5402   ...   (16 bits cada uno)
                    │  escala = max(|w|) / 7
                    ▼
INT4:     1       -3        7     ...   (4 bits cada uno) + 1 escala por grupo de 128 pesos
```
Al ejecutar, `peso ≈ entero × escala`. El error de redondeo es la pérdida de calidad.

### Memoria aproximada = nº parámetros × bytes por parámetro

| Formato | Bytes/param | 1.5B | 7B | 70B |
|---|---|---|---|---|
| FP32 | 4 | 6 GB | 28 GB | 280 GB |
| FP16 / BF16 | 2 | 3 GB | 14 GB | 140 GB |
| FP8 / INT8 | 1 | 1.5 GB | 7 GB | 70 GB |
| INT4 | 0.5 | 0.75 GB | 3.5 GB | 35 GB |

### La nomenclatura **WxAy**
- **W** = bits de los **pesos** (*weights*), **A** = bits de las **activaciones** (lo que fluye entre capas).
- **W4A16** (*weight-only*): pesos en INT4, cálculo en FP16. Ahorra mucha memoria. Ideal para **baja concurrencia / chat** (el cuello de botella es leer memoria).
- **W8A8** (FP8 o INT8): pesos **y** activaciones en 8 bits. Usa tensor cores de 8 bits → ideal para **alta concurrencia / producción**.

### Métodos (¿cómo se eligen los enteros?)
| Método | Idea | ¿Necesita datos de calibración? | Tiempo (1.5B) |
|---|---|---|---|
| **RTN** (*round-to-nearest*) | Redondear y ya | No | segundos |
| **FP8 dinámico** | FP8 por canal; escala de activaciones calculada al vuelo | No | ~1 min |
| **bitsandbytes** (NF4) | El mismo de QLoRA; se cuantiza al cargar | No | al cargar |
| **GPTQ** | Redondea capa a capa compensando el error con información de 2º orden (Hessiana) | Sí (~128–512 ejemplos) | 3–10 min |
| **AWQ** | Protege el ~1% de pesos "importantes" (según las activaciones) escalándolos | Sí | 3–10 min |
| **SmoothQuant** | Pasa la dificultad de las activaciones (outliers) a los pesos para poder usar INT8 en ambos | Sí | + GPTQ |
| **GGUF** (llama.cpp) | Formato para CPU / Ollama / LM Studio | Opcional (imatrix) | minutos |

### ¿Cuál elijo? (regla rápida)
```
¿Tienes GPU Ada/Hopper (L4, L40S, H100)?  ──sí──►  FP8 dinámico (casi sin pérdida, 2 min)
        │ no
        ▼
¿Necesitas meter el modelo en poca VRAM?  ──sí──►  W4A16 con GPTQ o AWQ
        │ no
        ▼
¿Muchos usuarios a la vez en T4/A100?     ──sí──►  W8A8 INT8 (SmoothQuant + GPTQ)
        │
        ▼
¿Solo quieres probar ya, sin crear nada?  ──────►  Modelo pre-cuantizado del Hub o bitsandbytes
```

---
## Alternativa A · La vía rápida: sin crear nada (≈15 min)

### A1 · Cuantización **al vuelo** dentro de vLLM
vLLM puede cuantizar el modelo **mientras lo carga**. Cero pasos previos, pero:
- No hay calibración → algo menos de calidad que GPTQ/AWQ.
- Se repite en cada arranque; no obtienes un modelo para compartir.

| Opción | GPU | Qué hace |
|---|---|---|
| `quantization="fp8"` | L4 / H100 (A100 solo pesos) | Pesos a FP8 + activaciones FP8 dinámicas |
| `quantization="bitsandbytes"` | **Cualquiera, incluida T4** | Pesos a 4 bits NF4 (como QLoRA) |

In [ ]:
# A1.a — FP8 al vuelo (solo si tu GPU es L4/H100; en A100 funciona como weight-only)
if cc >= (8, 0):
    !python /content/bench_vllm.py --model $MODEL_ID --quantization fp8 --tag "A1_fp8_al_vuelo" 2>&1 | grep -E "$FILTER"
else:
    print("⏭️ Tu GPU no soporta FP8 → salta a A1.b (bitsandbytes)")

In [ ]:
# A1.b — bitsandbytes 4 bits al vuelo (funciona en T4)
!python /content/bench_vllm.py --model $MODEL_ID --quantization bitsandbytes --tag "A1_bnb_4bit_al_vuelo" 2>&1 | grep -E "$FILTER"

### A2 · Descargar un modelo **ya cuantizado** del Hub
Muchos fabricantes publican versiones cuantizadas oficiales. vLLM **detecta el formato automáticamente**
leyendo `quantization_config` en el `config.json`. Busca en Hugging Face sufijos como
`-AWQ`, `-GPTQ-Int4`, `-FP8`, `-W4A16`, `-W8A8`, `-FP8-dynamic` (p. ej. la organización *RedHatAI* publica muchos).

In [ ]:
# A2 — Versión AWQ oficial de Qwen (no hace falta pasar quantization=...)
!python /content/bench_vllm.py --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --tag "A2_awq_hub" 2>&1 | grep -E "$FILTER"

---
## Alternativa B · FP8 dinámico con LLM Compressor (≈15 min) — *la más sencilla*

**El flujo de LLM Compressor siempre es el mismo (3 pasos):**
```
1. Cargar modelo (transformers)  →  2. oneshot(model, recipe [, dataset])  →  3. save_pretrained(save_compressed=True)
```
La **receta** (`recipe`) dice *qué* cuantizar y *cómo*:
- `targets="Linear"` → todas las capas lineales (atención y MLP), que concentran >95% de los pesos.
- `ignore=["lm_head"]` → la capa de salida se deja en FP16 (es muy sensible).
- `scheme="FP8_DYNAMIC"` → pesos FP8 por canal + activaciones FP8 por token calculadas al vuelo → **no necesita datos**.

> Puedes crearlo en cualquier GPU (incluida T4); para **ejecutarlo rápido** necesitas L4/H100.

In [ ]:
# ===========================================================================
# ALTERNATIVA B: FP8 dinámico (sin calibración)
# ===========================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

# 1) Cargar
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2) Receta + oneshot
recipe = QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC", ignore=["lm_head"])
oneshot(model=model, recipe=recipe)

# 3) Guardar en formato compressed-tensors
SAVE_FP8 = f"{WORK}/{MODEL_ID.split('/')[-1]}-FP8-Dynamic"
model.save_pretrained(SAVE_FP8, save_compressed=True)
tokenizer.save_pretrained(SAVE_FP8)
print("✅ Guardado en", SAVE_FP8)

# Liberar la GPU antes de lanzar vLLM
del model; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
!du -sh $SAVE_FP8
!python -c "import json; print(json.dumps(json.load(open('$SAVE_FP8/config.json'))['quantization_config'], indent=1)[:900])"

In [ ]:
!python /content/bench_vllm.py --model $SAVE_FP8 --tag "B_fp8_dynamic" 2>&1 | grep -E "$FILTER"

---
## Alternativa C · INT4 con GPTQ o AWQ (W4A16) (≈30 min) — *máxima compresión*

Aquí sí necesitamos **datos de calibración**: unos cientos de ejemplos parecidos a lo que verá el modelo en producción.
El algoritmo los pasa por el modelo, observa las activaciones de cada capa y elige el redondeo que **minimiza el error de salida**.

**Parámetros clave:**
| Parámetro | Valor de hoy | Producción | Efecto |
|---|---|---|---|
| `NUM_CALIBRATION_SAMPLES` | 128 | 256–512 | Más = mejor calidad, más lento |
| `MAX_SEQUENCE_LENGTH` | 512 | 2048 | Longitud de cada ejemplo |
| `scheme` | `W4A16` | `W4A16` | INT4 simétrico, grupos de 128 pesos |

> 💡 **Consejo de producción:** calibra con datos de tu dominio (tus tickets, tu código, tus documentos) y en el idioma de uso.

In [ ]:
# ===========================================================================
# ALTERNATIVA C (1/3): Dataset de calibración
# ===========================================================================
from datasets import load_dataset
from transformers import AutoTokenizer

NUM_CALIBRATION_SAMPLES = 128
MAX_SEQUENCE_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIBRATION_SAMPLES}]").shuffle(seed=42)

# Aplicamos la plantilla de chat: el modelo debe ver el mismo formato que verá en uso real
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)})
ds = ds.map(lambda s: tokenizer(s["text"], padding=False, max_length=MAX_SEQUENCE_LENGTH,
                                truncation=True, add_special_tokens=False),
            remove_columns=ds.column_names)
print(ds)

### C.1 · GPTQ
Elige **una** de las dos celdas (C.1 GPTQ o C.2 AWQ). Si te sobra tiempo, haz las dos y compara.

In [ ]:
# ===========================================================================
# ALTERNATIVA C (2/3): GPTQ W4A16  (~3-8 min en L4 para 1.5B)
# ===========================================================================
from transformers import AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
oneshot(model=model, dataset=ds, recipe=recipe,
        max_seq_length=MAX_SEQUENCE_LENGTH, num_calibration_samples=NUM_CALIBRATION_SAMPLES)

SAVE_GPTQ = f"{WORK}/{MODEL_ID.split('/')[-1]}-W4A16-GPTQ"
model.save_pretrained(SAVE_GPTQ, save_compressed=True)
tokenizer.save_pretrained(SAVE_GPTQ)
print("✅ Guardado en", SAVE_GPTQ)
del model; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
!du -sh $SAVE_GPTQ
!python /content/bench_vllm.py --model $SAVE_GPTQ --tag "C1_w4a16_gptq" 2>&1 | grep -E "$FILTER"

### C.2 · AWQ (alternativa a GPTQ)
AWQ no cambia el redondeo: **escala** los canales cuyos pesos multiplican activaciones grandes, para que sufran menos error.
Suele dar calidad igual o algo mejor que GPTQ en INT4. (Nota: el antiguo paquete `AutoAWQ` está archivado; hoy AWQ vive dentro de LLM Compressor.)

In [ ]:
# ===========================================================================
# ALTERNATIVA C (3/3): AWQ W4A16  (~3-8 min)
# ===========================================================================
from transformers import AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

recipe = AWQModifier(targets=["Linear"], scheme="W4A16_ASYM", ignore=["lm_head"])
oneshot(model=model, dataset=ds, recipe=recipe,
        max_seq_length=MAX_SEQUENCE_LENGTH, num_calibration_samples=NUM_CALIBRATION_SAMPLES)

SAVE_AWQ = f"{WORK}/{MODEL_ID.split('/')[-1]}-W4A16-AWQ"
model.save_pretrained(SAVE_AWQ, save_compressed=True)
tokenizer.save_pretrained(SAVE_AWQ)
print("✅ Guardado en", SAVE_AWQ)
del model; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
!du -sh $SAVE_AWQ
!python /content/bench_vllm.py --model $SAVE_AWQ --tag "C2_w4a16_awq" 2>&1 | grep -E "$FILTER"

---
## Alternativa D · INT8 W8A8 con SmoothQuant + GPTQ (≈10 min) — *la opción para T4 / A100*

Las activaciones tienen **outliers** (valores muy grandes en unos pocos canales) que arruinan INT8.
**SmoothQuant** divide esos canales de activación por un factor y multiplica los pesos correspondientes por el mismo factor:
el resultado matemático es idéntico, pero ahora ambos son "fáciles" de cuantizar.

Las recetas se pueden **encadenar** como una lista: primero suavizar, luego GPTQ.

In [ ]:
# ===========================================================================
# ALTERNATIVA D: W8A8 INT8 (SmoothQuant + GPTQ)
# ===========================================================================
from transformers import AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

recipe = [
    SmoothQuantModifier(smoothing_strength=0.8),
    GPTQModifier(targets="Linear", scheme="W8A8", ignore=["lm_head"]),
]
oneshot(model=model, dataset=ds, recipe=recipe,
        max_seq_length=MAX_SEQUENCE_LENGTH, num_calibration_samples=NUM_CALIBRATION_SAMPLES)

SAVE_INT8 = f"{WORK}/{MODEL_ID.split('/')[-1]}-W8A8-INT8"
model.save_pretrained(SAVE_INT8, save_compressed=True)
tokenizer.save_pretrained(SAVE_INT8)
print("✅ Guardado en", SAVE_INT8)
del model; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
!du -sh $SAVE_INT8
!python /content/bench_vllm.py --model $SAVE_INT8 --tag "D_w8a8_int8" 2>&1 | grep -E "$FILTER"

---
## Paso 8 · Comparar resultados

Cada ejecución de `bench_vllm.py` añadió una fila a `/content/resultados.jsonl`.
Completa la columna **VRAM** con el valor `Model loading took X GiB` que viste en cada log.

In [ ]:
# ===========================================================================
# PASO 8: Tabla comparativa
# ===========================================================================
import pandas as pd, subprocess, os
df = pd.read_json("/content/resultados.jsonl", lines=True)
base = df.loc[df.tag == "0_base_fp16", "tok_s"]
if len(base):
    df["speedup_vs_base"] = (df.tok_s / base.iloc[0]).round(2)

def disk(m):
    if os.path.isdir(m):
        return subprocess.run(["du", "-sh", m], capture_output=True, text=True).stdout.split()[0]
    return "(Hub/al vuelo)"
df["disco"] = df.model.map(disk)
df["mates_ok(=7)"] = df.respuesta_mates.str.contains("7")
df[["tag", "disco", "load_s", "tok_s", "speedup_vs_base", "mates_ok(=7)"]]

### (Opcional) Calidad objetiva con `lm-eval`
Las 4 preguntas son una prueba "a ojo". Para medir de verdad se usa un benchmark. GSM8K (problemas de matemáticas)
con 100 ejemplos tarda ~3–5 min por modelo:

In [ ]:
# !uv pip install --system -q "lm-eval[vllm]"
# for m in [MODEL_ID, SAVE_FP8, SAVE_GPTQ]:
#     !lm_eval --model vllm --model_args pretrained={m},max_model_len=2048,gpu_memory_utilization=0.8 \
#              --tasks gsm8k --num_fewshot 5 --limit 100 --batch_size auto 2>&1 | tail -n 6

---
## Paso 9 · Servir el modelo cuantizado como API (compatible con OpenAI)

En producción no usamos `LLM(...)` en Python: levantamos un **servidor**. Cualquier cliente de OpenAI
(LangChain, LlamaIndex, Cursor, tu app...) funciona cambiando solo la `base_url`.

In [ ]:
# ===========================================================================
# PASO 9a: Arrancar el servidor en segundo plano (elige tu mejor modelo)
# ===========================================================================
import subprocess, time, requests
MEJOR = SAVE_GPTQ if 'SAVE_GPTQ' in globals() else MODEL_ID   # cámbialo por el que prefieras
extra = ["--dtype", "half"] if cc < (8, 0) else []
server = subprocess.Popen(["vllm", "serve", MEJOR, "--served-model-name", "mi-modelo",
                           "--port", "8000", "--max-model-len", "2048", *extra],
                          stdout=open("/content/vllm_server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("✅ Servidor listo"); break
    except Exception:
        pass
    time.sleep(3)
else:
    print("❌ No arrancó; revisa /content/vllm_server.log")

In [ ]:
# PASO 9b: Consumirlo con el cliente oficial de OpenAI
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="no-hace-falta")
r = client.chat.completions.create(
    model="mi-modelo",
    messages=[{"role": "user", "content": "Dame 3 ventajas de cuantizar un LLM."}],
    temperature=0.2, max_tokens=200)
print(r.choices[0].message.content)

In [ ]:
%%bash
# PASO 9c: Lo mismo con curl (celda bash para que IPython no interprete las llaves del JSON)
curl -s http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" \
  -d '{"model":"mi-modelo","messages":[{"role":"user","content":"Hola, ¿quién eres?"}],"max_tokens":60}' | python -m json.tool

In [ ]:
# 🧹 Parar el servidor al terminar
server.terminate(); server.wait(); print("Servidor detenido")

---
## (Opcional) Publicar tu modelo en Hugging Face
```python
from huggingface_hub import login, HfApi
login()  # token con permiso de escritura
HfApi().create_repo("tu-usuario/Qwen2.5-1.5B-Instruct-W4A16", exist_ok=True)
HfApi().upload_folder(folder_path=SAVE_GPTQ, repo_id="tu-usuario/Qwen2.5-1.5B-Instruct-W4A16")
```
Después cualquiera lo ejecuta con `vllm serve tu-usuario/Qwen2.5-1.5B-Instruct-W4A16`.

## (Opcional) GGUF para CPU / Ollama
vLLM está pensado para GPU. Si el destino es un portátil o Ollama (como en la sesión anterior), el formato es **GGUF**:
```bash
git clone --depth 1 https://github.com/ggml-org/llama.cpp && pip install -r llama.cpp/requirements.txt
python llama.cpp/convert_hf_to_gguf.py <carpeta_modelo_hf> --outfile modelo-f16.gguf
# llama-quantize se obtiene compilando llama.cpp (cmake) o de sus releases
./llama-quantize modelo-f16.gguf modelo-Q4_K_M.gguf Q4_K_M
```

---
## 🆘 Problemas frecuentes
| Error | Causa | Solución |
|---|---|---|
| `CUDA out of memory` | Otro proceso tiene la GPU (p. ej. un modelo cargado en el notebook o el servidor) | Ejecuta las celdas de `del model ...`, para el servidor o reinicia la sesión |
| `Bfloat16 is only supported on GPUs with compute capability >= 8.0` | T4 | Ya lo maneja el script con `dtype="half"` |
| `fp8 ... not supported` / sin kernel | GPU antigua | Usa W8A8 INT8 (Alt. D) o bitsandbytes (Alt. A1.b) |
| W4A16 va **más lento** que la base en T4 | No hay kernel Marlin para sm75 | Normal: en T4 elige INT8 o bnb; Marlin necesita sm80+ |
| Imports de `llmcompressor` fallan | Versión distinta | `from llmcompressor import oneshot` es la API actual (antes `llmcompressor.transformers`) |
| Error de versiones tras instalar | No reiniciaste la sesión | `Entorno de ejecución → Reiniciar sesión` |

---
## 📝 Entrega
1. La **tabla del Paso 8** (captura) con al menos: base, una alternativa A y una de B/C/D.
2. Responde en 5–6 líneas:
   - ¿Cuánto se redujo el tamaño en disco y la VRAM de pesos con cada técnica?
   - ¿Cambió la velocidad? ¿Por qué crees que en tu GPU fue así?
   - ¿Notaste pérdida de calidad en alguna respuesta?
   - Si tuvieras que desplegar un modelo de 8B para 50 usuarios en una L4 de 24 GB, ¿qué técnica elegirías y por qué?